# Qwen 14B Optimized Notebook

Lucas Harrich

This notebook keeps the same basic flow as `model_qwen_1_5b_optimized.ipynb`, but upgrades the model to a much stronger Qwen model above 7B parameters.

It reads prompts from `dataset_clean.csv`, runs them through `Qwen/Qwen3-14B`, and saves the results in `submission_qwen_14b.csv`.

Important Colab notes:
- use a GPU runtime
- the model is loaded in 4-bit to fit more easily on Colab GPUs
- `useSample = True` is recommended for the first test run

Short model info:
- Model: `Qwen/Qwen3-14B`
- Size: about `14.8B` parameters
- Load mode here: `4-bit` quantized with `bitsandbytes`
- Input length here: `600` tokens
- Output length here: up to `180` new tokens


In [ ]:
# Keep Colab-managed packages untouched and install only what this notebook needs.
%pip install -q "transformers>=4.51.0,<4.58.0" "accelerate>=1.4.0" "bitsandbytes>=0.45.0" sentencepiece


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 10.7 MB/s eta 0:00:00


In [ ]:
# Imports and basic settings
from pathlib import Path as pathClass
import pandas as pandasLib
import torch
from IPython.display import display as showTable
from transformers import AutoTokenizer as autoTokenizerClass
from transformers import AutoModelForCausalLM as autoModelForCausalLmClass
from transformers import BitsAndBytesConfig as bitsAndBytesConfigClass

# Input and output files
inputPath = pathClass("dataset_clean.csv")
outputPath = pathClass("submission_qwen_14b.csv")

# Lightweight run settings
useSample = False
sampleSize = 5
modelName = "Qwen/Qwen3-14B"
maxInputTokens = 600
maxNewTokens = 180
enableThinking = False

# Instruction prefix for every prompt
systemPrompt = (
    "Beantworte die oesterreichische Steuerfrage auf Deutsch. "
    "Sei kurz, praezise und erfinde keine Fakten."
)

# Required input schema
requiredColumns = {"id", "prompt"}

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print(f"model: {modelName}")
print(f"cuda available: {torch.cuda.is_available()}")


model: Qwen/Qwen3-14B
cuda available: True


In [ ]:
# Load and validate the input data
sourceTable = pandasLib.read_csv(inputPath, encoding="utf-8-sig")

missingColumns = requiredColumns - set(sourceTable.columns)
if missingColumns:
    raise ValueError(f"Missing columns: {sorted(missingColumns)}")

sourceTable["prompt"] = sourceTable["prompt"].fillna("").astype(str)

workingTable = sourceTable.head(sampleSize).copy() if useSample else sourceTable.copy()
print(f"rows to process: {len(workingTable)}")
showTable(workingTable.head())


rows to process: 643


,id,prompt
0,CORP-TAX-001,Was ist die steuerliche Bemessungsgrundlage fü...
1,CORP-TAX-002,"Welche steuerlichen Konsequenzen hat es, wenn ..."
2,CORP-TAX-003,"Welche Körperschaften sind verpflichtet, sämtl..."
3,CORP-TAX-004,Wie wird eine Dividende einer österreichischen...
4,CORP-TAX-005,Was unterscheidet die steuerliche Behandlung e...


In [ ]:
# GPU check and model loading
from shutil import which as whichFunc
import subprocess

if whichFunc("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("nvidia-smi not found. This usually means the notebook is not running on a GPU runtime.")

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU found. In Colab go to Runtime > Change runtime type > T4 GPU or another GPU, save, then rerun the notebook from the top."
    )

def pickComputeDtype():
    majorVersion, _ = torch.cuda.get_device_capability(0)
    return torch.bfloat16 if majorVersion >= 8 else torch.float16

def loadGeneratorBundle():
    computeDtype = pickComputeDtype()
    quantConfig = bitsAndBytesConfigClass(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=computeDtype
    )

    activeTokenizer = autoTokenizerClass.from_pretrained(
        modelName,
        use_fast=True
    )
    if activeTokenizer.pad_token is None:
        activeTokenizer.pad_token = activeTokenizer.eos_token

    activeModel = autoModelForCausalLmClass.from_pretrained(
        modelName,
        device_map="auto",
        quantization_config=quantConfig,
        torch_dtype=computeDtype
    )
    activeModel.eval()
    return activeTokenizer, activeModel, computeDtype

def cleanText(value):
    return " ".join(str(value).strip().split())

tokenizer, model, computeDtype = loadGeneratorBundle()
print(f"loaded model: {modelName}")
print(f"compute dtype: {computeDtype}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model-00001-of-00008.safetensors:   0%|          | 0.00/3.84G [00:00<?, ?B/s]

model-00003-of-00008.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

loaded model: Qwen/Qwen3-14B
compute dtype: torch.float16


In [ ]:
# Prompt building and generation
def buildPromptText(sourcePrompt):
    promptText = cleanText(sourcePrompt)
    if not promptText:
        return ""

    messages = [
        {"role": "system", "content": systemPrompt},
        {"role": "user", "content": promptText}
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=enableThinking
    )

def generateAnswer(sourcePrompt):
    fullPrompt = buildPromptText(sourcePrompt)
    if not fullPrompt:
        return ""

    tokenBatch = tokenizer(
        fullPrompt,
        return_tensors="pt",
        truncation=True,
        max_length=maxInputTokens
    )
    tokenBatch = {name: value.to("cuda") for name, value in tokenBatch.items()}

    with torch.inference_mode():
        generatedIds = model.generate(
            **tokenBatch,
            max_new_tokens=maxNewTokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.05,
            use_cache=True
        )

    answerIds = generatedIds[0][tokenBatch["input_ids"].shape[1]:]
    answerText = tokenizer.decode(answerIds, skip_special_tokens=True)
    return cleanText(answerText)

testPrompt = "Was ist Google Colab und wofuer nutzt man es?"
print(generateAnswer(testPrompt))


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Google Colab (Colaboratory) ist eine kostenlose Cloud-Plattform von Google, die es ermöglicht, Python-Code in einem interaktiven Notebook-Umgebung auszuführen. Es wird häufig für maschinelles Lernen, Datenanalyse und experimentelle Programmierung genutzt.


In [ ]:
# Generate answers and save the output CSV
answerList = []

for rowIndex, row in enumerate(workingTable.itertuples(index=False), start=1):
    answerText = generateAnswer(row.prompt)
    answerList.append({"id": row.id, "answer": answerText})
    print(f"{rowIndex}/{len(workingTable)}: {row.id}")

resultTable = pandasLib.DataFrame(answerList, columns=["id", "answer"])
resultTable.to_csv(outputPath, index=False, encoding="utf-8")

print(f"saved: {outputPath.resolve()}")
showTable(resultTable.head())


1/643: CORP-TAX-001
2/643: CORP-TAX-002
3/643: CORP-TAX-003
4/643: CORP-TAX-004
5/643: CORP-TAX-005
6/643: CORP-TAX-006
7/643: CORP-TAX-007
8/643: CORP-TAX-008
9/643: CORP-TAX-009
10/643: CORP-TAX-010
11/643: CORP-TAX-011
12/643: CORP-TAX-012
13/643: CORP-TAX-013
14/643: CORP-TAX-014
15/643: CORP-TAX-015
16/643: CORP-TAX-016
17/643: CORP-TAX-017
18/643: CORP-TAX-018
19/643: CORP-TAX-019
20/643: CORP-TAX-020
21/643: CORP-TAX-021
22/643: CORP-TAX-022
23/643: CORP-TAX-023
24/643: CORP-TAX-024
25/643: CORP-TAX-025
26/643: CORP-TAX-026
27/643: CORP-TAX-027
28/643: CORP-TAX-028
29/643: CORP-TAX-029
30/643: CORP-TAX-030
31/643: CORP-TAX-031
32/643: CORP-TAX-032
33/643: CORP-TAX-033
34/643: CORP-TAX-034
35/643: CORP-TAX-035
36/643: CORP-TAX-036
37/643: CORP-TAX-037
38/643: CORP-TAX-038
39/643: CORP-TAX-039
40/643: CORP-TAX-040
41/643: CORP-TAX-041
42/643: CORP-TAX-042
43/643: CORP-TAX-043
44/643: CORP-TAX-044
45/643: CORP-TAX-045
46/643: CORP-TAX-046
47/643: CORP-TAX-047
48/643: CORP-TAX-048
4

,id,answer
0,CORP-TAX-001,Die steuerliche Bemessungsgrundlage für die Kö...
1,CORP-TAX-002,Wenn eine österreichische Körperschaft ein **z...
2,CORP-TAX-003,"In Österreich sind **Gewerbebetriebe**, die al..."
3,CORP-TAX-004,(a) **Bei der österreichischen Tochtergesellsc...
4,CORP-TAX-005,Die steuerliche Behandlung von **offenen** und...
